# Example for using custom Hamiltonian function

In [ ]:
import matplotlib.pyplot as plt
import jax.numpy as jnp
from cthree.quantity import Quantity

In [ ]:
amplitude = Quantity(
    value=jnp.array(1.55e8),
    min_value=jnp.array(0.0),
    max_value=jnp.array(1e9),
    unit="Hz",
    name="Amplitude",
    two_pi=True,
)

frequency = Quantity(
    value=jnp.array(4.8e9 * 2 * jnp.pi),
    min_value=jnp.array(0.8 * 4.8e9 * 2 * jnp.pi),
    max_value=jnp.array(1.2 * 4.8e9 * 2 * jnp.pi),
    unit="Hz",
    name="lo_freq",
    two_pi=True,
)


def cos_envelope(t, amp, freq):
    """Define a cosine envelope."""
    return amp * jnp.cos(freq * t)


def TLS_hamiltonian(t, amp, freq):
    """Define a Two level system Hamiltonian."""
    return 0.5 * FREQ * sigma_z + sigma_x * cos_envelope(t, amp, freq)


def grad_amp(t, amp, freq):
    """Gradient of Hamiltonian wrt amplitude."""
    return sigma_x * jnp.cos(freq * t)


def grad_frequency(t, amp, freq):
    """Gradient of Hamiltonian wrt frequency."""
    return -sigma_x * amp * t * jnp.sin(freq * t)


sigma_x = jnp.array([[0j, 1], [1, 0]])
sigma_z = jnp.diag(jnp.array([1.0, -1.0]))
FREQ = 4.8e9 * 2 * jnp.pi

In [ ]:
from cthree.model.custom_hamiltonian import CustomHamiltonian
from cthree.model.closed_system import ClosedSystem

TLS = CustomHamiltonian(
    hamiltonian_function=TLS_hamiltonian,
    parameters=[amplitude, frequency],
    gradient_functions=[grad_amp, grad_frequency],
)
model = ClosedSystem(TLS)

In [ ]:
TLS.dimension()

In [ ]:
from cthree.measurement.state_transfer_fidelity import StateTransferFidelity
from cthree.propagation.scipy_expm_goat import ScipyExpmGOAT

t_final = 10e-9

prop = ScipyExpmGOAT(model, res=100e9)

init = jnp.array([[1.0], [0]])  # |0>
target = jnp.array([[0.0], [1]])  # |1>
zeroone = StateTransferFidelity(
    propagation=prop,
    initial_state=init,
    target_state=target,
    times=jnp.array([0.0, t_final]),
)

In [ ]:
def make_plot():
    """Plot the signal."""
    ts = jnp.linspace(0, t_final, 1001)
    states = jnp.reshape(prop.propagate(ts), (-1, 2))
    sig = cos_envelope(ts, amp=amplitude.get_value()[0], freq=frequency.get_value()[0])

    fig, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts / 1e-9, sig)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts / 1e-9, jnp.abs(states) ** 2)
    ax[1].set_ylabel("Population")
    ax[-1].set_xlabel("Time [ns]")
    return fig, ax


make_plot()

In [ ]:
zeroone.measure()

In [ ]:
from cthree.optimisation_map import OptimisationMap
from cthree.optimisers.scipy_optimiser_gradient import ScipyOptimiserGradient

optmap = OptimisationMap()
optmap.add(TLS)
opt = ScipyOptimiserGradient(zeroone, optimisables=optmap)

In [ ]:
opt.optimise()

In [ ]:
make_plot()